# Tree ensemble formulations

*Note: the formulations shown in this notebook are **experimental**: their
names, keyword arguments and variables may change in future releases.*

*All the models in this notebook have fewer than 2000 variables and 2000
constraints, so they can be solved with a size-limited license.*

By default, Gurobi Machine Learning formulates each tree of a random forest or
of a gradient boosting regressor on its own (the `"leaf"` formulation). Three
formulations from the literature can be selected instead with the `formulation`
keyword argument of `add_predictor_constr`.
`"misic"`, `"misic_lazy"` and `"ocean"` model the whole ensemble with variables
shared by all trees; `"biggs_perakis"` models each tree on its own, like
`"leaf"`, with a tighter linear relaxation:

| `formulation` | Reference |
|---|---|
| `"leaf"` (default) | one binary variable per leaf, indicator constraints |
| `"misic"` | Mišić, *Optimization of Tree Ensembles*, Operations Research (2020) |
| `"misic_lazy"` | `"misic"` with lazy split constraints |
| `"ocean"` | Parmentier & Vidal, *Optimal Counterfactual Explanations in Tree Ensembles*, ICML (2021), as in their OCEAN implementation |
| `"biggs_perakis"` | Biggs & Perakis, *Tight Mixed-Integer Optimization Formulations for Prescriptive Trees*, Machine Learning (2025) |

The formulations are described in the
[Mixed Integer Formulations](https://gurobi-optimization-gurobi-machine-learning.readthedocs-hosted.com/en/latest/user/mip-models.html)
page of the user guide. In this notebook we compare them on a small gradient
boosting regressor: their size, the solutions they return, and how to read
which leaf each tree selected.

In [ ]:
import gurobipy as gp
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.ensemble import GradientBoostingRegressor

from gurobi_ml import add_predictor_constr

FORMULATIONS = ["leaf", "misic", "ocean", "biggs_perakis"]

## Train the regressor

We use the diabetes dataset of scikit-learn: ten (normalized) features of a
patient and a measure of disease progression one year later. We fit a gradient
boosting regressor with 20 trees of depth 3.

In [ ]:
X, y = load_diabetes(return_X_y=True)
feature_lb = X.min(axis=0)
feature_ub = X.max(axis=0)

regression = GradientBoostingRegressor(n_estimators=20, max_depth=3, random_state=0)
regression.fit(X, y)

## The optimization model

The optimization problem is as simple as possible: find the input, within the
range of values observed in the data, that maximizes the predicted disease
progression. The function below builds this model with a given formulation.

In [ ]:
def build_model(formulation):
    m = gp.Model()
    m.Params.OutputFlag = 0
    x = m.addMVar((1, X.shape[1]), lb=feature_lb, ub=feature_ub, name="x")
    y_pred = m.addMVar((1, 1), lb=-gp.GRB.INFINITY, name="y")
    pred_constr = add_predictor_constr(
        m, regression, x, y_pred, formulation=formulation
    )
    m.setObjective(y_pred.sum(), gp.GRB.MAXIMIZE)
    m.update()
    return m, pred_constr, x

## Model size

`print_stats` shows how each formulation is built. For the default `"leaf"`
formulation, it lists each tree:

In [ ]:
m, pred_constr, x = build_model("leaf")
pred_constr.print_stats()

`"misic"` and `"ocean"` are built for the whole ensemble at once, with
variables shared by all trees, so `print_stats` only shows the totals. In
`"misic"`, all binary variables are shared split variables, one per distinct
threshold of the ensemble; the leaf variables of the trees are continuous:

In [ ]:
m, pred_constr, x = build_model("misic")
pred_constr.print_stats()

The table below compares the sizes of the formulations. Indicator
constraints are counted separately from linear constraints.

In [ ]:
sizes = {}
for formulation in FORMULATIONS:
    m, pred_constr, x = build_model(formulation)
    sizes[formulation] = {
        "variables": m.NumVars,
        "binaries": m.NumBinVars,
        "linear constraints": m.NumConstrs,
        "indicator constraints": m.NumGenConstrs,
    }
pd.DataFrame(sizes).T

The number of binary variables grows differently with the number of trees. In
`"leaf"` and `"biggs_perakis"` there is one binary variable per leaf of each
tree. In `"misic"` there is one per distinct threshold, and trees fitted on the
same data reuse many thresholds. We only build the models here, without solving
them, so the larger ensembles can be built with a size-limited license too.

In [ ]:
growth = {}
for n_trees in (10, 20, 40, 80, 160):
    ensemble = GradientBoostingRegressor(
        n_estimators=n_trees, max_depth=3, random_state=0
    ).fit(X, y)
    row = {}
    for formulation in ("leaf", "misic", "biggs_perakis"):
        with gp.Model() as m:
            m.Params.OutputFlag = 0
            x = m.addMVar((1, X.shape[1]), lb=feature_lb, ub=feature_ub)
            add_predictor_constr(m, ensemble, x, formulation=formulation)
            m.update()
            row[formulation] = m.NumBinVars
    growth[n_trees] = row
pd.DataFrame(growth).T.rename_axis("trees")

## Solve with each formulation

We now solve the model with each formulation. The last column gives the
prediction of the regression at the optimal value of $x$.

In [ ]:
solutions = {}
results = {}
for formulation in FORMULATIONS:
    m, pred_constr, x = build_model(formulation)
    m.optimize()
    solutions[formulation] = (pred_constr, x.X.copy())
    results[formulation] = {
        "objective": m.ObjVal,
        "predict(x)": regression.predict(x.X)[0],
    }
pd.DataFrame(results).T

The formulations do not agree, and none of the objective values equals the
prediction at $x$. Both effects come from the split thresholds.

The optimal $x$ lies exactly on some split thresholds. With the default
`epsilon=0`, the constraints of both branches of a split are satisfied at the
threshold, and the solver may choose either branch. `predict` then evaluates
$x$ with its own rule, which may go down the other branch. This is why the
prediction at $x$ is not a good check of the solution.

The formulations also differ at a threshold. In `"misic"`, one shared binary
variable decides the direction of a split for all trees. In `"leaf"`, `"ocean"` and `"biggs_perakis"`, each tree
decides for itself, and different trees may take different branches at the
same threshold. The resulting combination of leaves may be reached by no input
at all.

## Read the selected leaves

To check a solution, we read which leaf each tree selected. All formulations
expose the leaf variables in the `tree_leaves` attribute: one entry per tree,
with a matrix variable `variables` (one row per input, one column per leaf that
can be reached) and the node ids `nodes` of those leaves in the scikit-learn
tree.

Below, `leaf_box` computes the box of inputs reaching a leaf of a scikit-learn
tree. The inputs reaching a leaf satisfy $lo < x \le hi$ componentwise.

In [ ]:
def leaf_box(tree, leaf):
    lo = np.full(tree.n_features, -np.inf)
    hi = np.full(tree.n_features, np.inf)
    parent = {}
    for node in range(tree.node_count):
        if tree.children_left[node] >= 0:
            parent[tree.children_left[node]] = (node, True)
            parent[tree.children_right[node]] = (node, False)
    node = leaf
    while node in parent:
        node, went_left = parent[node]
        feature, threshold = tree.feature[node], tree.threshold[node]
        if went_left:
            hi[feature] = min(hi[feature], threshold)
        else:
            lo[feature] = max(lo[feature], threshold)
    return lo, hi

For a solution, we intersect the boxes of the selected leaves of all trees with
the bounds of the input variables. If the intersection is empty, no input
reaches this combination of leaves: the objective value is not attained by the
regression. Otherwise, we evaluate the regression at the middle of the
intersection.

In [ ]:
trees = [estimator[0].tree_ for estimator in regression.estimators_]

checks = {}
for formulation, (pred_constr, x_opt) in solutions.items():
    lo, hi = feature_lb.copy(), feature_ub.copy()
    lo_strict = np.zeros(X.shape[1], dtype=bool)
    for tree, leaves in zip(trees, pred_constr.tree_leaves):
        selected = leaves.nodes[np.argmax(leaves.variables.X[0])]
        box_lo, box_hi = leaf_box(tree, selected)
        lo_strict |= box_lo >= lo
        lo = np.maximum(lo, box_lo)
        hi = np.minimum(hi, box_hi)
    reachable = bool(np.all(np.where(lo_strict, lo < hi, lo <= hi)))
    checks[formulation] = {
        "objective": results[formulation]["objective"],
        "leaves reachable": reachable,
        "predict(inside)": (
            regression.predict(((lo + hi) / 2).reshape(1, -1))[0]
            if reachable
            else np.nan
        ),
    }
pd.DataFrame(checks).T

For `"misic"`, the selected leaves are reached by inputs, and the regression evaluated at such an input gives the optimal
objective value. For the other formulations, the selected leaves are not
compatible: their objective value is larger than any value the regression can
predict.

The leaf variables can also be used in constraints, for example to restrict
the solution to leaves that contain enough training data.

## Lazy split constraints

The `"misic_lazy"` formulation is `"misic"` with the constraints linking the
leaf variables to the split variables marked lazy (their Gurobi `Lazy`
attribute is set to 3). Gurobi then keeps them out of the relaxation and adds
them when they are violated, including to cut off the root relaxation.

In [ ]:
m, pred_constr, x = build_model("misic_lazy")

lazy = m.getAttr("Lazy", m.getConstrs())
print(f"{sum(level == 3 for level in lazy)} of {len(lazy)} linear constraints are lazy")

The feasible solutions are the same as with `"misic"`, and so is the optimal
value.

In [ ]:
m.optimize()
print(f"misic:       {results['misic']['objective']:.4f}")
print(f"misic_lazy:  {m.ObjVal:.4f}")

Whether lazy split constraints make the solution faster depends on the
problem. They usually help when few branch-and-bound nodes are needed, and can
be slower when the search branches a lot. The models of this notebook solve
in a fraction of a second with every formulation, so they cannot tell the
formulations apart in speed.

Copyright © 2023-2026 Gurobi Optimization, LLC